# Colab SSH Bootstrap

Run all cells to start an SSH tunnel into this Colab runtime.

**Order:** colab-ssh runs **before** the Google Drive mount so the `trycloudflare.com` hostname usually appears **before** the Drive consent dialog. That way browser automation (and you) can read the hostname without Drive OAuth blocking the first code cell.

Connect from Cursor after the hostname line appears:
```
scripts/connect_colab.sh <HOSTNAME>
```

The Drive cell is for a persistent clone under My Drive. If you skip Drive access, the next cell uses `/content/recsys_playground` (not persisted across sessions).

In [ ]:
!pip install colab-ssh --upgrade -q
import os, secrets, subprocess
from google.colab import userdata

# Read NTFY_TOPIC from Colab Secrets manager (set once: left panel → key icon)
_NTFY_TOPIC = userdata.get('NTFY_TOPIC')

# Random SSH password — password auth will be disabled after key is installed
_SSH_PASSWORD = secrets.token_urlsafe(32)

from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password=_SSH_PASSWORD)

# Install SSH public key and disable password auth
_PUBKEY = "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIKUpXNTTzI8MXt8QCwY0agMVQTEOJ9Nu1Aqq4nFkrrYM colab-automation"
os.makedirs('/root/.ssh', mode=0o700, exist_ok=True)
authorized = '/root/.ssh/authorized_keys'
keys = open(authorized).read() if os.path.exists(authorized) else ''
if _PUBKEY not in keys:
    with open(authorized, 'a') as f:
        f.write(_PUBKEY + '
')
os.chmod(authorized, 0o600)
subprocess.run(['sed', '-i', 's/^#*PasswordAuthentication.*/PasswordAuthentication no/', '/etc/ssh/sshd_config'], check=True)
subprocess.run(['sed', '-i', 's/^#*PubkeyAuthentication.*/PubkeyAuthentication yes/', '/etc/ssh/sshd_config'], check=True)
subprocess.run(['service', 'ssh', 'restart'], check=True)

# Relay hostname to agent via ntfy.sh
from colab_ssh.get_tunnel_config import get_argo_tunnel_config
import urllib.request
_info = get_argo_tunnel_config()
urllib.request.urlopen(urllib.request.Request(
    f'https://ntfy.sh/{_NTFY_TOPIC}',
    data=_info['domain'].encode(), method='POST'
))
print(f"Hostname relayed: {_info['domain']}")
print("SSH key auth only — password auth disabled.")

In [ ]:
# Mount Google Drive for persistent storage (runs after SSH so hostname is available first)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

if os.path.isdir('/content/drive/MyDrive'):
    WORK_DIR = '/content/drive/MyDrive/colab/recsys_playground'
else:
    WORK_DIR = '/content/recsys_playground'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'

if not os.path.exists(repo_dir):
    !git clone {repo_url}

%cd {repo_dir}
!git pull origin main
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn requests papermill